In [6]:
from dotenv import load_dotenv

load_dotenv()

from typing import Any, List, Optional, Dict
import os

from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ChatMessage,
)
from langchain_core.outputs import ChatGeneration, ChatResult
from openai import OpenAI
from pydantic import Field, PrivateAttr


class QwenChatModel(BaseChatModel):
    """
    基于 BaseChatModel 封装的阿里云 Qwen 自定义类。
    支持 enable_thinking 参数以获取思考过程。
    """

    model_name: str = Field(default="qwen-plus", alias="model")
    api_key: Optional[str] = Field(default=None)
    base_url: str = Field(default="https://dashscope.aliyuncs.com/compatible-mode/v1")
    enable_thinking: bool = Field(default=True, description="是否开启思考过程")
    temperature: float = 0.7


    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.api_key = self.api_key or os.getenv("DASHSCOPE_API_KEY")
        if not self.api_key:
            raise ValueError("API Key is required. Set DASHSCOPE_API_KEY env var or pass it in.")

        # 初始化 OpenAI 原生客户端
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
        )

    @property
    def _llm_type(self) -> str:
        return "qwen_dashscope"

    def _convert_messages_to_openai_format(self, messages: List[BaseMessage]) -> List[Dict[str, Any]]:
        """将 LangChain 消息格式转换为 OpenAI SDK 要求的格式"""
        openai_messages = []
        for msg in messages:
            role = "user"
            if isinstance(msg, HumanMessage):
                role = "user"
            elif isinstance(msg, AIMessage):
                role = "assistant"
            elif isinstance(msg, SystemMessage):
                role = "system"
            elif isinstance(msg, ChatMessage):
                role = msg.role

            openai_messages.append({"role": role, "content": msg.content})
        return openai_messages

    def _generate(
            self,
            messages: List[BaseMessage],
            stop: Optional[List[str]] = None,
            run_manager: Optional[CallbackManagerForLLMRun] = None,
            **kwargs: Any,
    ) -> ChatResult:
        """核心生成逻辑"""

        # 1. 转换消息格式
        openai_messages = self._convert_messages_to_openai_format(messages)

        # 2. 准备请求参数 (extra_body 是关键)
        extra_body = kwargs.get("extra_body", {})
        if self.enable_thinking:
            extra_body["enable_thinking"] = True

        # 3. 调用 OpenAI SDK
        response = self._client.chat.completions.create(
            model=self.model_name,
            messages=openai_messages,
            temperature=self.temperature,
            extra_body=extra_body,  # 传入百炼特有参数
            stop=stop,
            **kwargs
        )

        # 4. 解析结果
        choice = response.choices[0]
        message = choice.message
        content = message.content

        # 5. 处理思考过程 (Reasoning Content)
        # 百炼 API 的思考内容通常在 message 的 reasoning_content 字段中（如果有）
        # 或者有时在 extra_fields 里，我们将其放入 additional_kwargs 以便后续查看
        reasoning_content = getattr(message, "reasoning_content", None)

        additional_kwargs = {}
        if reasoning_content:
            additional_kwargs["reasoning_content"] = reasoning_content

        # 6. 构造 LangChain 的返回值
        generations = [
            ChatGeneration(
                message=AIMessage(
                    content=content,
                    additional_kwargs=additional_kwargs  # 思考过程存在这里
                )
            )
        ]

        return ChatResult(generations=generations)

In [7]:
llm = QwenChatModel(
    model="qwen-plus",  # 或者 qwen-max
    enable_thinking=True,
    temperature=0.5
)

In [8]:
from langchain_core.messages import HumanMessage
from IPython.display import display, Markdown

messages = [
    HumanMessage(content="请帮我解方程：x^2 - 5x + 6 = 0")
]
result = llm.invoke(messages)

reasoning = result.additional_kwargs.get("reasoning_content", "")
final_answer = result.content

# 渲染思考过程
print("=== 思考过程 ===")
if reasoning:
    display(Markdown(f"> {reasoning}"))  # 加个引用符号区分
else:
    print("无思考过程")

print("\n=== 最终回答 ===")
display(Markdown(final_answer))

=== 思考过程 ===


> 我现在要解这个二次方程x² - 5x + 6 = 0。首先，我记得解二次方程的方法有几种，比如因式分解、求根公式、配方法。这里可能先试试因式分解，因为系数比较简单，可能容易分解。

首先，二次项的系数是1，所以因式分解的话应该是(x + a)(x + b) = 0的形式，展开后是x² + (a + b)x + ab = 0。现在原方程是x² - 5x + 6 = 0，所以需要找到两个数a和b，使得a + b = -5（因为一次项系数是-5），而ab = 6（常数项是6）。

不过这里可能需要注意符号，因为原方程的一次项是-5x，所以应该是(x - m)(x - n) = 0，这样展开后是x² - (m + n)x + mn = 0，所以m + n = 5，mn = 6。这样更准确，因为原方程的一次项系数是-5，所以两个根的和是5，积是6。

现在找两个数，和为5，积为6。可能的数对有1和6（和为7，不对），2和3（和为5，积为6），对了，2和3。所以可以分解为(x - 2)(x - 3) = 0。

接下来，根据零乘积定理，如果两个数的乘积为0，那么至少其中一个数为0。所以x - 2 = 0或者x - 3 = 0，解得x = 2或x = 3。

为了验证是否正确，可以把x=2代入原方程：2² - 5*2 + 6 = 4 - 10 + 6 = 0，对的。x=3代入：3² - 5*3 + 6 = 9 - 15 + 6 = 0，也对。

另外，也可以用求根公式来验证。二次方程ax² + bx + c = 0的根是x = [-b ± √(b² - 4ac)]/(2a)。这里a=1，b=-5，c=6，所以判别式D = (-5)² - 4*1*6 = 25 - 24 = 1，根号D=1。所以x = [5 ± 1]/2，即(5+1)/2=3，(5-1)/2=2，和因式分解的结果一致。

所以这个方程的解是x=2和x=3。


=== 最终回答 ===


要解方程 $x^2 - 5x + 6 = 0$，可以采用**因式分解法**或**求根公式法**。以下是详细步骤：

---

### **方法一：因式分解**
1. 观察方程 $x^2 - 5x + 6 = 0$，尝试将其分解为两个一次因式的乘积：
   $$
   (x - a)(x - b) = 0
   $$
   展开后为 $x^2 - (a + b)x + ab = 0$。

2. 比较系数：
   - 一次项系数：$-(a + b) = -5 \Rightarrow a + b = 5$
   - 常数项：$ab = 6$

3. 找到满足条件的整数对：  
   $a = 2$，$b = 3$（因为 $2 + 3 = 5$ 且 $2 \times 3 = 6$）。

4. 因式分解结果：
   $$
   (x - 2)(x - 3) = 0
   $$

5. 解得：
   $$
   x - 2 = 0 \quad \text{或} \quad x - 3 = 0 \Rightarrow x = 2 \quad \text{或} \quad x = 3
   $$

---

### **方法二：求根公式**
对于一般二次方程 $ax^2 + bx + c = 0$，根为：
$$
x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}
$$
代入 $a = 1$，$b = -5$，$c = 6$：
$$
x = \frac{-(-5) \pm \sqrt{(-5)^2 - 4 \cdot 1 \cdot 6}}{2 \cdot 1} = \frac{5 \pm \sqrt{25 - 24}}{2} = \frac{5 \pm 1}{2}
$$
计算得：
$$
x = \frac{5 + 1}{2} = 3 \quad \text{或} \quad x = \frac{5 - 1}{2} = 2
$$

---

### **验证**
将 $x = 2$ 和 $x = 3$ 代入原方程：
- $2^2 - 5 \cdot 2 + 6 = 4 - 10 + 6 = 0$
- $3^2 - 5 \cdot 3 + 6 = 9 - 15 + 6 = 0$

两者均满足方程。

---

### **最终答案**
$$
\boxed{x = 2 \quad \text{或} \quad x = 3}
$$